# QC re-run of `xmens_inputs` vs the pipeline's QC-passing objects

Compares the objects from re-applying QC to the `xmens_inputs` files (`qc_passed_<species>.h5ad` from the
`verify_qc.sh` run, in `RERUN`) with the pipeline's QC-passing objects (`qc_passed_<species>.h5ad` in `PIPELINE`).
For each species:

1. **Nuclei**: per sample, the nuclei in each object, shared, and in only one of them.
2. **Data on the shared nuclei**: whether `X`, `layers["counts"]`, `obsm["spatial"]`, the gene table and every
   cell-table column are identical.

Loads the full objects, one species at a time (about 2.2 GB each); about 10 GB of memory at peak. Kernel: `rsc_uv`.

In [1]:
RERUN = "/n/netscratch/eddy_lab/Everyone/nhilgert/2026_11_05_qc_verify_pinned"
PIPELINE = "/n/lab_storage/eddy_lab/users/nhilgert/xmens_final/data/interim"

In [2]:
import gc
import os

os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")

import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp

EXPECTED_TO_DIFFER = {"doubletdetection_doublet_score"}  # DoubletDetection is not reproducible between runs


def n_different(a, b):
    """Number of entries that differ between two matrices of the same shape."""
    if sp.issparse(a) or sp.issparse(b):
        return int((sp.csr_matrix(a) != sp.csr_matrix(b)).nnz)
    return int((np.asarray(a) != np.asarray(b)).sum())


def same_column(a, b):
    """Whether two columns are identical, and for numbers the largest difference."""
    if pd.api.types.is_numeric_dtype(a) and pd.api.types.is_numeric_dtype(b):
        x, y = a.to_numpy(dtype=float), b.to_numpy(dtype=float)
        d = np.abs(x - y)
        d = d[~np.isnan(d)]
        return np.array_equal(x, y, equal_nan=True), f"largest difference {d.max() if d.size else 0:.3g}"
    n = int((a.astype(str).to_numpy() != b.astype(str).to_numpy()).sum())
    return n == 0, f"{n:,} values differ"

In [3]:
def compare(species):
    new = ad.read_h5ad(f"{RERUN}/qc_passed_{species}.h5ad")
    old = ad.read_h5ad(f"{PIPELINE}/qc_passed_{species}.h5ad")
    assert new.obs_names.is_unique and old.obs_names.is_unique

    # 1. Nuclei per sample
    rows = []
    for sample in pd.unique(pd.concat([old.obs["sample"].astype(str), new.obs["sample"].astype(str)])):
        a = new.obs_names[new.obs["sample"].astype(str) == sample]
        b = old.obs_names[old.obs["sample"].astype(str) == sample]
        n = len(a.intersection(b))
        rows.append({"sample": sample, "re-run": len(a), "pipeline": len(b), "shared": n,
                     "only re-run": len(a) - n, "only pipeline": len(b) - n})
    nuclei = pd.DataFrame(rows).set_index("sample")
    nuclei.loc["total"] = nuclei.sum()
    nuclei["% of pipeline shared"] = (100 * nuclei["shared"] / nuclei["pipeline"]).round(2)

    # 2. Data on the shared nuclei, in the same order in both objects
    shared = old.obs_names.intersection(new.obs_names)
    new, old = new[shared], old[shared]
    checks = []
    def add(what, same, detail):
        checks.append({"what": what, "identical": bool(same), "detail": detail})

    add("genes (names and order)", new.var_names.equals(old.var_names), f"{new.n_vars:,} vs {old.n_vars:,} genes")
    for col in old.var.columns.intersection(new.var.columns):
        add(f"gene table: {col}", *same_column(new.var[col], old.var[col]))
    n = n_different(new.X, old.X)
    add("X", n == 0, f"{n:,} of {old.X.nnz:,} nonzero entries differ")
    for key in sorted(set(old.layers) & set(new.layers)):
        n = n_different(new.layers[key], old.layers[key])
        add(f"layers['{key}']", n == 0, f"{n:,} of {old.layers[key].nnz:,} nonzero entries differ")
    for key in sorted(set(old.obsm) & set(new.obsm)):
        x, y = np.asarray(new.obsm[key], dtype=float), np.asarray(old.obsm[key], dtype=float)
        n = int((~((x == y) | (np.isnan(x) & np.isnan(y)))).reshape(len(y), -1).any(axis=1).sum())
        add(f"obsm['{key}']", n == 0, f"{n:,} of {len(y):,} rows differ")
    for col in old.obs.columns.intersection(new.obs.columns):
        same, detail = same_column(new.obs[col], old.obs[col])
        add(f"cell table: {col}", same, detail + (" (expected to differ)" if col in EXPECTED_TO_DIFFER else ""))
    only = sorted(set(old.obs.columns) ^ set(new.obs.columns))
    add("cell-table columns present in both", not only, f"in only one object: {only}" if only else "all")
    checks = pd.DataFrame(checks).set_index("what")

    print(f"{species}: {len(shared):,} shared nuclei; differences on them, apart from those expected:",
          [w for w, r in checks.iterrows() if not r["identical"] and "expected" not in r["detail"]] or "none")
    del new, old
    gc.collect()
    return nuclei, checks

In [4]:
nuclei, checks = compare("mouse")
display(nuclei)
display(checks)

mouse: 67,635 shared nuclei; differences on them, apart from those expected: none


,re-run,pipeline,shared,only re-run,only pipeline,% of pipeline shared
sample,,,,,,
uninduced_mck8a,11819,11892,11763,56,129,98.92
uninduced_mck8b,8455,8446,8419,36,27,99.68
1_dpi_early_mck1,3456,3458,3455,1,3,99.91
1_dpi_late_mck12a,7510,7497,7479,31,18,99.76
1_dpi_late_mck12b,6856,6860,6818,38,42,99.39
2_dpi_mck15,5084,5014,4998,86,16,99.68
3_dpi_mck10a,8351,8273,8192,159,81,99.02
3_dpi_mck10b,8262,8196,8137,125,59,99.28
4_dpi_bleeding_mck11a,8420,8433,8374,46,59,99.30


,identical,detail
what,,
genes (names and order),True,"33,698 vs 33,698 genes"
gene table: gene_ids,True,0 values differ
gene table: feature_types,True,0 values differ
gene table: genome,True,0 values differ
gene table: mt,True,largest difference 0
gene table: ribo,True,largest difference 0
gene table: hb,True,largest difference 0
X,True,"0 of 134,458,577 nonzero entries differ"
layers['counts'],True,"0 of 134,458,577 nonzero entries differ"


In [5]:
nuclei, checks = compare("human")
display(nuclei)
display(checks)

human: 70,121 shared nuclei; differences on them, apart from those expected: none


,re-run,pipeline,shared,only re-run,only pipeline,% of pipeline shared
sample,,,,,,
secretory_mck6a,12989,13024,12962,27,62,99.52
secretory_mck6b,12659,12685,12622,37,63,99.50
secretory_mck6c,7516,7517,7483,33,34,99.55
menstrual_mck5a,12796,12794,12767,29,27,99.79
menstrual_mck5b,12801,12795,12753,48,42,99.67
menstrual_mck5c,11591,11548,11534,57,14,99.88
total,70352,70363,70121,231,242,99.66


,identical,detail
what,,
genes (names and order),True,"36,601 vs 36,601 genes"
gene table: gene_ids,True,0 values differ
gene table: feature_types,True,0 values differ
gene table: genome,True,0 values differ
gene table: mt,True,largest difference 0
gene table: ribo,True,largest difference 0
gene table: hb,True,largest difference 0
X,True,"0 of 138,196,896 nonzero entries differ"
layers['counts'],True,"0 of 138,196,896 nonzero entries differ"


**Reading the results**

- Ingest and the threshold QC are deterministic, so the same nuclei reach DoubletDetection in both runs.
  DoubletDetection is not reproducible between runs, so a few nuclei per sample are called doublets in only one run;
  they make up **only re-run** and **only pipeline**.
- On the shared nuclei every check should read `True`, except the doublet score.